# Formative 3: DQN Hyperparameter Experiments
## Mariam Awini Issah - Lower Parameters Tuning

**Experiment Strategy**: Lower/Conservative Parameters
- **Focus**: Test effect of reducing learning rate, gamma, and batch size
- **Hypothesis**: Lower parameters → more stable training, slower convergence
- **Goal**: Find stability-performance trade-off

### Baseline Configuration (Reference):
```
learning_rate: 1e-4
gamma: 0.99
batch_size: 32
exploration_final_eps: 0.01
buffer_size: 50000
target_update_interval: 1000
```

In [17]:
# Import required libraries
import ale_py
import gymnasium as gym
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from stable_baselines3 import DQN
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack
import warnings
warnings.filterwarnings('ignore')

# Register Atari environments
gym.register_envs(ale_py)

print("✓ All dependencies imported successfully!")
print("Team Member: Mariam Awini Issah")
print("Experiment Type: Lower Parameters Tuning")

✓ All dependencies imported successfully!
Team Member: Mariam Awini Issah
Experiment Type: Lower Parameters Tuning


In [18]:
# Baseline Configuration (Reference)
BASELINE_CONFIG = {
    "policy": "CnnPolicy",
    "learning_rate": 1e-4,
    "buffer_size": 50000,
    "learning_starts": 10000,
    "batch_size": 32,
    "tau": 1.0,
    "gamma": 0.99,
    "train_freq": 4,
    "gradient_steps": 1,
    "exploration_fraction": 0.1,
    "exploration_final_eps": 0.01,
    "target_update_interval": 1000,
    "verbose": 1,
}

print("Baseline Config:")
for k, v in BASELINE_CONFIG.items():
    print(f"  {k:.<30} {v}")

Baseline Config:
  policy........................ CnnPolicy
  learning_rate................. 0.0001
  buffer_size................... 50000
  learning_starts............... 10000
  batch_size.................... 32
  tau........................... 1.0
  gamma......................... 0.99
  train_freq.................... 4
  gradient_steps................ 1
  exploration_fraction.......... 0.1
  exploration_final_eps......... 0.01
  target_update_interval........ 1000
  verbose....................... 1


In [19]:
def train_and_evaluate_dqn(config, member_name, exp_num, total_timesteps=50000, num_eval_episodes=3):
    """
    Train and evaluate a DQN model with given configuration.
    
    Args:
        config: Hyperparameter configuration
        member_name: Name of team member
        exp_num: Experiment number
        total_timesteps: Training duration
        num_eval_episodes: Episodes for evaluation
    
    Returns:
        dict with results
    """
    print(f"\n{'='*70}")
    print(f"EXPERIMENT {exp_num} - {member_name}")
    print(f"{'='*70}")
    
    # Print configuration
    print("\nConfiguration (Modified from Baseline):")
    for key, value in config.items():
        baseline_val = BASELINE_CONFIG.get(key)
        status = "↓" if isinstance(value, (int, float)) and baseline_val and value < baseline_val else "→"
        print(f"  {key:.<30} {value:>12} {status}")
    
    model_name = f"dqn_{member_name.replace(' ', '_')}_exp{exp_num}"
    
    try:
        # Create environment
        env = make_atari_env("ALE/PrivateEye-v5", n_envs=1, seed=0)
        env = VecFrameStack(env, n_stack=4)
        
        # Train model
        print(f"\n🔄 Training for {total_timesteps} timesteps...")
        model = DQN(
            policy=config["policy"],
            env=env,
            learning_rate=config["learning_rate"],
            buffer_size=config["buffer_size"],
            learning_starts=config["learning_starts"],
            batch_size=config["batch_size"],
            tau=config["tau"],
            gamma=config["gamma"],
            train_freq=config["train_freq"],
            gradient_steps=config["gradient_steps"],
            exploration_fraction=config["exploration_fraction"],
            exploration_final_eps=config["exploration_final_eps"],
            target_update_interval=config["target_update_interval"],
            verbose=0,
        )
        
        model.learn(total_timesteps=total_timesteps)
        model.save(model_name)
        print(f"✓ Model saved as {model_name}.zip")
        
        # Evaluate
        print(f"\n📊 Evaluating on {num_eval_episodes} episodes...")
        rewards = []
        
        for ep in range(num_eval_episodes):
            obs, _ = env.reset()
            episode_reward = 0
            done = False
            
            while not done:
                action, _ = model.predict(obs, deterministic=True)
                obs, reward, terminated, truncated, _ = env.step(action)
                episode_reward += reward
                done = terminated or truncated
            
            rewards.append(episode_reward)
            print(f"  Episode {ep+1}: Reward = {episode_reward:.0f}")
        
        env.close()
        
        avg_reward = np.mean(rewards)
        max_reward = np.max(rewards)
        min_reward = np.min(rewards)
        
        results = {
            "exp_num": exp_num,
            "lr": config["learning_rate"],
            "gamma": config["gamma"],
            "batch_size": config["batch_size"],
            "epsilon_final": config["exploration_final_eps"],
            "buffer_size": config["buffer_size"],
            "avg_reward": avg_reward,
            "max_reward": max_reward,
            "min_reward": min_reward,
            "model_name": model_name,
            "status": "✓ Success"
        }
        
        print(f"\n✓ Results:")
        print(f"  Average Reward: {avg_reward:.2f}")
        print(f"  Max Reward: {max_reward:.2f}")
        print(f"  Min Reward: {min_reward:.2f}")
        
        return results
    
    except Exception as e:
        print(f"\n✗ Error: {str(e)}")
        return {
            "exp_num": exp_num,
            "avg_reward": 0,
            "max_reward": 0,
            "status": f"Failed: {str(e)}"
        }

print("✓ Training function defined")

✓ Training function defined


## Lower Parameters Tuning Experiments

**Strategy**: Conservative/Stable Learning
- Exp 1-3: Lower learning rates
- Exp 4-6: Lower gamma values  
- Exp 7-8: Smaller batch sizes
- Exp 9: Lower exploration
- Exp 10: Combined optimal lower params

### Run Your Experiments Below

In [20]:
# EXPERIMENT 1: Very Low Learning Rate (5e-5)
# Hypothesis: Slower but more stable learning

exp1_config = {**BASELINE_CONFIG, 
               "learning_rate": 5e-5}  # 50% reduction from baseline

print("EXPERIMENT 1: Very Low Learning Rate")
print(f"Learning Rate: 1e-4 → 5e-5 (50% reduction)")
print("Hypothesis: Slower convergence but more stable training\n")

# result_exp1 = train_and_evaluate_dqn(exp1_config, "Mariam Awini Issah", 1)
# Uncomment above line to run the experiment

EXPERIMENT 1: Very Low Learning Rate
Learning Rate: 1e-4 → 5e-5 (50% reduction)
Hypothesis: Slower convergence but more stable training



In [21]:
# EXPERIMENT 2: Extra Low Learning Rate (1e-5)
# Hypothesis: Extreme stability, very slow convergence

exp2_config = {**BASELINE_CONFIG, 
               "learning_rate": 1e-5}  # 90% reduction

print("EXPERIMENT 2: Extra Low Learning Rate")
print(f"Learning Rate: 1e-4 → 1e-5 (90% reduction)")
print("Hypothesis: Extremely stable but very slow learning\n")

# result_exp2 = train_and_evaluate_dqn(exp2_config, "Mariam Awini Issah", 2)
# Uncomment above line to run the experiment

EXPERIMENT 2: Extra Low Learning Rate
Learning Rate: 1e-4 → 1e-5 (90% reduction)
Hypothesis: Extremely stable but very slow learning



In [22]:
# EXPERIMENT 3: Low Learning Rate + Low Buffer
# Hypothesis: Combined conservative approach

exp3_config = {**BASELINE_CONFIG,
               "learning_rate": 5e-5,
               "buffer_size": 30000}  # 40% buffer reduction

print("EXPERIMENT 3: Low Learning Rate + Reduced Buffer")
print(f"Learning Rate: 1e-4 → 5e-5, Buffer: 50000 → 30000")
print("Hypothesis: Stable learning with more focused memory\n")

# result_exp3 = train_and_evaluate_dqn(exp3_config, "Mariam Awini Issah", 3)
# Uncomment above line to run the experiment

EXPERIMENT 3: Low Learning Rate + Reduced Buffer
Learning Rate: 1e-4 → 5e-5, Buffer: 50000 → 30000
Hypothesis: Stable learning with more focused memory



In [23]:
# EXPERIMENT 4: Lower Gamma (0.95)
# Hypothesis: Focus on short-term rewards

exp4_config = {**BASELINE_CONFIG,
               "gamma": 0.95}  # Down from 0.99

print("EXPERIMENT 4: Lower Gamma (Discount Factor)")
print(f"Gamma: 0.99 → 0.95")
print("Hypothesis: More focus on immediate rewards, less on future\n")

# result_exp4 = train_and_evaluate_dqn(exp4_config, "Mariam Awini Issah", 4)
# Uncomment above line to run the experiment

EXPERIMENT 4: Lower Gamma (Discount Factor)
Gamma: 0.99 → 0.95
Hypothesis: More focus on immediate rewards, less on future



In [24]:
# EXPERIMENT 5: Very Low Gamma (0.90)
# Hypothesis: Strong focus on immediate rewards

exp5_config = {**BASELINE_CONFIG,
               "gamma": 0.90}  # Down from 0.99

print("EXPERIMENT 5: Very Low Gamma")
print(f"Gamma: 0.99 → 0.90")
print("Hypothesis: Strong emphasis on immediate rewards\n")

# result_exp5 = train_and_evaluate_dqn(exp5_config, "Mariam Awini Issah", 5)
# Uncomment above line to run the experiment

EXPERIMENT 5: Very Low Gamma
Gamma: 0.99 → 0.90
Hypothesis: Strong emphasis on immediate rewards



In [25]:
# EXPERIMENT 6: Low Gamma + Low Learning Rate
# Hypothesis: Combined conservative approach

exp6_config = {**BASELINE_CONFIG,
               "gamma": 0.95,
               "learning_rate": 5e-5}

print("EXPERIMENT 6: Low Gamma + Low Learning Rate")
print(f"Gamma: 0.99 → 0.95, LR: 1e-4 → 5e-5")
print("Hypothesis: Double stability boost\n")

# result_exp6 = train_and_evaluate_dqn(exp6_config, "Mariam Awini Issah", 6)
# Uncomment above line to run the experiment

EXPERIMENT 6: Low Gamma + Low Learning Rate
Gamma: 0.99 → 0.95, LR: 1e-4 → 5e-5
Hypothesis: Double stability boost



In [26]:
# EXPERIMENT 7: Small Batch Size (16)
# Hypothesis: More frequent, noisier updates

exp7_config = {**BASELINE_CONFIG,
               "batch_size": 16}  # 50% reduction

print("EXPERIMENT 7: Small Batch Size")
print(f"Batch Size: 32 → 16")
print("Hypothesis: Smaller updates = more diverse learning\n")

# result_exp7 = train_and_evaluate_dqn(exp7_config, "Mariam Awini Issah", 7)
# Uncomment above line to run the experiment

EXPERIMENT 7: Small Batch Size
Batch Size: 32 → 16
Hypothesis: Smaller updates = more diverse learning



In [27]:
# EXPERIMENT 8: Very Small Batch Size (8)
# Hypothesis: Extreme noisiness

exp8_config = {**BASELINE_CONFIG,
               "batch_size": 8}  # 75% reduction

print("EXPERIMENT 8: Very Small Batch Size")
print(f"Batch Size: 32 → 8")
print("Hypothesis: Very noisy but diverse updates\n")

# result_exp8 = train_and_evaluate_dqn(exp8_config, "Mariam Awini Issah", 8)
# Uncomment above line to run the experiment

EXPERIMENT 8: Very Small Batch Size
Batch Size: 32 → 8
Hypothesis: Very noisy but diverse updates



In [28]:
# EXPERIMENT 9: Low Exploration Final Epsilon (0.001)
# Hypothesis: Very little exploration after training

exp9_config = {**BASELINE_CONFIG,
               "exploration_final_eps": 0.001}  # Down from 0.01

print("EXPERIMENT 9: Very Low Exploration Final Epsilon")
print(f"Epsilon Final: 0.01 → 0.001")
print("Hypothesis: Minimal exploration, exploitation focused\n")

# result_exp9 = train_and_evaluate_dqn(exp9_config, "Mariam Awini Issah", 9)
# Uncomment above line to run the experiment

EXPERIMENT 9: Very Low Exploration Final Epsilon
Epsilon Final: 0.01 → 0.001
Hypothesis: Minimal exploration, exploitation focused



In [29]:
# EXPERIMENT 10: Combined Optimal Lower Parameters
# Hypothesis: Best combination of stability factors

exp10_config = {**BASELINE_CONFIG,
                "learning_rate": 5e-5,      # Low LR
                "gamma": 0.95,              # Lower gamma
                "batch_size": 16,           # Smaller batch
                "exploration_final_eps": 0.001,  # Low exploration
                "buffer_size": 30000}       # Smaller buffer

print("EXPERIMENT 10: Combined Optimal Lower Parameters")
print("Configuration:")
print("  Learning Rate: 1e-4 → 5e-5")
print("  Gamma: 0.99 → 0.95")
print("  Batch Size: 32 → 16")
print("  Epsilon Final: 0.01 → 0.001")
print("  Buffer Size: 50000 → 30000")
print("Hypothesis: Best combination of lower/stable parameters\n")

# result_exp10 = train_and_evaluate_dqn(exp10_config, "Mariam Awini Issah", 10)
# Uncomment above line to run the experiment

EXPERIMENT 10: Combined Optimal Lower Parameters
Configuration:
  Learning Rate: 1e-4 → 5e-5
  Gamma: 0.99 → 0.95
  Batch Size: 32 → 16
  Epsilon Final: 0.01 → 0.001
  Buffer Size: 50000 → 30000
Hypothesis: Best combination of lower/stable parameters



## Results Compilation

After running all 10 experiments, fill in this table:

In [30]:
# Create results tracking dataframe
results = pd.DataFrame({
    'Exp': list(range(1, 11)),
    'Learning_Rate': [5e-5, 1e-5, 5e-5, np.nan, np.nan, 5e-5, np.nan, np.nan, np.nan, 5e-5],
    'Gamma': [np.nan, np.nan, np.nan, 0.95, 0.90, 0.95, np.nan, np.nan, np.nan, 0.95],
    'Batch_Size': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 16, 8, np.nan, 16],
    'Epsilon_Final': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.001, 0.001],
    'Avg_Reward': [None]*10,
    'Max_Reward': [None]*10,
    'Notes': [
        'Very low LR',
        'Extra low LR',
        'Low LR + low buffer',
        'Lower gamma',
        'Very low gamma',
        'Low gamma + low LR',
        'Small batch',
        'Very small batch',
        'Low exploration',
        'Combined optimal'
    ]
})

print("EXPERIMENT RESULTS TRACKING")
print("="*80)
print(results.to_string(index=False))
print("\n  Fill in Avg_Reward and Max_Reward after running experiments")

EXPERIMENT RESULTS TRACKING
 Exp  Learning_Rate  Gamma  Batch_Size  Epsilon_Final Avg_Reward Max_Reward               Notes
   1        0.00005    NaN         NaN            NaN       None       None         Very low LR
   2        0.00001    NaN         NaN            NaN       None       None        Extra low LR
   3        0.00005    NaN         NaN            NaN       None       None Low LR + low buffer
   4            NaN   0.95         NaN            NaN       None       None         Lower gamma
   5            NaN   0.90         NaN            NaN       None       None      Very low gamma
   6        0.00005   0.95         NaN            NaN       None       None  Low gamma + low LR
   7            NaN    NaN        16.0            NaN       None       None         Small batch
   8            NaN    NaN         8.0            NaN       None       None    Very small batch
   9            NaN    NaN         NaN          0.001       None       None     Low exploration
  10        

## Analysis & Key Insights

### Questions to Answer After Experiments:

1. **Stability Impact**: Did lower parameters result in more stable training compared to baseline?
2. **Performance Trade-off**: What was the trade-off between stability and performance?
3. **Best Lower Configuration**: Which experiment (1-10) showed the best reward?
4. **Convergence Speed**: Which lower parameter reduced convergence speed the most?

### For Presentation (2 minutes):
- Show your top 3 results
- Compare against baseline
- Explain the most surprising finding
- Recommend the best "lower parameters" configuration

In [31]:
# Visualization (fill in after collecting results)
def plot_results(results_df):
    """Plot Mariam's lower parameters experiment results"""
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Plot 1: Rewards across experiments
    if 'Avg_Reward' in results_df.columns:
        axes[0, 0].bar(results_df['Exp'], results_df['Avg_Reward'], color='skyblue')
        axes[0, 0].set_title("Average Reward Across Experiments")
        axes[0, 0].set_xlabel("Experiment Number")
        axes[0, 0].set_ylabel("Avg Reward")
        axes[0, 0].axhline(y=BASELINE_CONFIG.get('gamma', 0), color='r', linestyle='--', label='Baseline')
    
    # Plot 2: Comparison of key parameters
    axes[0, 1].scatter(results_df['Exp'], results_df['Avg_Reward'], s=100, alpha=0.6)
    axes[0, 1].set_title("Rewards vs Experiment")
    axes[0, 1].set_xlabel("Experiment Number")
    axes[0, 1].set_ylabel("Avg Reward")
    
    # Plot 3: Notes summary
    axes[1, 0].axis('off')
    summary_text = "LOWER PARAMETERS FOCUS:\n" + "\n".join([f"Exp {i}: {note}" for i, note in enumerate(results_df['Notes'], 1)])
    axes[1, 0].text(0.1, 0.5, summary_text, fontsize=10, verticalalignment='center', family='monospace')
    
    # Plot 4: Best/Worst
    if 'Avg_Reward' in results_df.columns and results_df['Avg_Reward'].notna().any():
        best_idx = results_df['Avg_Reward'].idxmax()
        axes[1, 1].text(0.5, 0.8, f"Best Experiment: #{results_df.loc[best_idx, 'Exp']}", 
                       ha='center', fontsize=12, weight='bold')
        axes[1, 1].text(0.5, 0.6, f"Reward: {results_df.loc[best_idx, 'Avg_Reward']:.2f}", 
                       ha='center', fontsize=11)
        axes[1, 1].axis('off')
    
    plt.tight_layout()
    plt.show()

# Uncomment after running all experiments and filling in rewards:
# plot_results(results)